# Transformer Phase Model

This notebook implements the first GRTPL transformer baseline from the real HC-3 phase-target pipeline.

The model sees only historical decimated causal phase tokens. For every input sample, the target is the nominal causal phase that corresponds to the next future acausal 180 degree theta target time.

This notebook is intentionally not executed in this commit because the local GPU is busy. It also does not install PyTorch automatically, so the AMD Windows PyTorch build is not overwritten by a generic CPU wheel.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

project_root

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd

from grtpl.hc3 import download_buzsaki_hc3_session_files, load_eeg_channels, read_hc3_xml_metadata
from grtpl.signal import acausal_bandpass, causal_bandpass, decimate_by_timestamp, hilbert_phase
from grtpl.targets import find_next_phase_crossings, next_crossing_for_samples, nominal_phase_targets

In [ ]:
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
except ImportError as exc:
    raise ImportError(
        "PyTorch is required for this notebook. Install the AMD Windows PyTorch build first; "
        "do not use a generic dependency sync that replaces it with a CPU wheel."
    ) from exc

torch.__version__

## Configuration

The first baseline uses the small `ec013.544` HC-3 session and the same adjacent channel pair as the exploration notebook: channels 38 and 39, with differential LFP defined as `channel_39 - channel_38`.

The final transformer input rate starts at 25 Hz. That is near the strict Nyquist rate for a 6-10 Hz theta band while still leaving enough samples for a useful historical context.

In [ ]:
@dataclass(frozen=True)
class PhaseModelConfig:
    topdir: str = "ec013.33"
    session: str = "ec013.544"
    selected_channels: tuple[int, int] = (38, 39)
    theta_band_hz: tuple[float, float] = (6.0, 10.0)
    nominal_frequency_hz: float = 8.0
    transformer_input_rate_hz: float = 25.0
    max_target_horizon_s: float = 0.5
    context_length: int = 128
    vocab_size: int = 360
    batch_size: int = 64
    train_fraction: float = 0.8
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 4
    dropout: float = 0.1
    learning_rate: float = 3e-4
    max_steps: int = 1000
    eval_every: int = 100


cfg = PhaseModelConfig()
cfg

## Build Phase Token Table

This cell recreates the real-data target table from raw HC-3 LFP. The target table is the bridge between signal processing and the transformer:

- `phase_token`: rounded causal phase token in `[0, 359]` at the decimated input time.
- `target_nominal_phase_rad`: nominal phase target for the future acausal 180 degree mark.
- `target_time_s`: future timestamp of the acausal 180 degree target.

In [ ]:
def build_phase_target_table(cfg: PhaseModelConfig) -> pd.DataFrame:
    raw_dir = project_root / "data" / "raw"
    download_buzsaki_hc3_session_files(cfg.topdir, cfg.session, raw_dir, extensions=("xml", "eeg"))

    session_dir = raw_dir / "hc3" / cfg.topdir / cfg.session
    xml_path = session_dir / f"{cfg.session}.xml"
    eeg_path = session_dir / f"{cfg.session}.eeg"

    metadata = read_hc3_xml_metadata(xml_path)
    sample_rate_hz = metadata["lfp_sampling_rate_hz"]
    n_channels = metadata["n_channels"]

    channel_lfp = load_eeg_channels(eeg_path, n_channels=n_channels, channels=list(cfg.selected_channels))
    differential_lfp = channel_lfp[:, 1] - channel_lfp[:, 0]
    timestamps = np.arange(len(differential_lfp)) / sample_rate_hz

    causal_theta, _, _ = causal_bandpass(
        differential_lfp,
        sample_rate_hz,
        band_hz=cfg.theta_band_hz,
        order=4,
    )
    reference_theta = acausal_bandpass(
        differential_lfp,
        sample_rate_hz,
        band_hz=cfg.theta_band_hz,
        order=4,
    )

    causal_phase = hilbert_phase(causal_theta)
    reference_phase = hilbert_phase(reference_theta)
    input_timestamps, input_phase = decimate_by_timestamp(
        timestamps,
        causal_phase,
        cfg.transformer_input_rate_hz,
    )

    crossing_timestamps, _ = find_next_phase_crossings(
        timestamps,
        reference_phase,
        target_phase_rad=np.pi,
    )
    target_timestamps = next_crossing_for_samples(
        input_timestamps,
        crossing_timestamps,
        max_horizon_s=cfg.max_target_horizon_s,
    )
    target_phase = nominal_phase_targets(
        input_phase,
        input_timestamps,
        target_timestamps,
        nominal_frequency_hz=cfg.nominal_frequency_hz,
    )

    causal_phase_deg = np.rad2deg(input_phase) % 360
    phase_token = np.floor(causal_phase_deg).astype(np.int64)
    table = pd.DataFrame(
        {
            "input_time_s": input_timestamps,
            "phase_token": phase_token,
            "causal_phase_deg": causal_phase_deg,
            "target_time_s": target_timestamps,
            "target_nominal_phase_rad": target_phase,
            "target_nominal_phase_deg": np.rad2deg(target_phase) % 360,
            "lead_time_ms": (target_timestamps - input_timestamps) * 1000,
        }
    )
    return table.dropna().reset_index(drop=True)


target_table = build_phase_target_table(cfg)
target_table.head()

In [ ]:
target_table[["input_time_s", "phase_token", "target_time_s", "target_nominal_phase_deg", "lead_time_ms"]].describe()

## Windowed Dataset

Each example is a fixed-length historical sequence of phase tokens. The label is represented as `(cos(target_phase), sin(target_phase))`, which avoids the 0/360 wrap discontinuity while still reporting errors in degrees.

In [ ]:
class PhaseWindowDataset(Dataset):
    def __init__(self, table: pd.DataFrame, context_length: int):
        if len(table) <= context_length:
            raise ValueError(f"Need more rows than context_length={context_length}; got {len(table)}")
        self.tokens = torch.as_tensor(table["phase_token"].to_numpy(), dtype=torch.long)
        self.target_phase = torch.as_tensor(
            table["target_nominal_phase_rad"].to_numpy(),
            dtype=torch.float32,
        )
        self.input_time = torch.as_tensor(table["input_time_s"].to_numpy(), dtype=torch.float32)
        self.target_time = torch.as_tensor(table["target_time_s"].to_numpy(), dtype=torch.float32)
        self.context_length = context_length

    def __len__(self) -> int:
        return len(self.tokens) - self.context_length + 1

    def __getitem__(self, idx: int):
        end = idx + self.context_length
        phase = self.target_phase[end - 1]
        target_vec = torch.stack([torch.cos(phase), torch.sin(phase)])
        return {
            "tokens": self.tokens[idx:end],
            "target_vec": target_vec,
            "target_phase": phase,
            "input_time": self.input_time[end - 1],
            "target_time": self.target_time[end - 1],
        }


split_idx = int(len(target_table) * cfg.train_fraction)
train_table = target_table.iloc[:split_idx].reset_index(drop=True)
val_table = target_table.iloc[max(0, split_idx - cfg.context_length + 1):].reset_index(drop=True)

train_ds = PhaseWindowDataset(train_table, cfg.context_length)
val_ds = PhaseWindowDataset(val_table, cfg.context_length)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=False)

len(train_ds), len(val_ds)

## Model

The input phase is treated as a token in a 360-token vocabulary. A learned token embedding and learned positional embedding feed a causal Transformer encoder. The final hidden state predicts a unit vector for the nominal phase target.

In [ ]:
class PhaseTokenTransformer(nn.Module):
    def __init__(self, cfg: PhaseModelConfig):
        super().__init__()
        self.context_length = cfg.context_length
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.position_embedding = nn.Embedding(cfg.context_length, cfg.d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=4 * cfg.d_model,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg.n_layers)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, 2)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = tokens.shape
        if seq_len > self.context_length:
            raise ValueError(f"seq_len={seq_len} exceeds context_length={self.context_length}")

        positions = torch.arange(seq_len, device=tokens.device)
        x = self.token_embedding(tokens) + self.position_embedding(positions)[None, :, :]
        causal_mask = torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=tokens.device),
            diagonal=1,
        )
        x = self.encoder(x, mask=causal_mask)
        final_state = self.norm(x[:, -1, :])
        phase_vec = self.head(final_state)
        return nn.functional.normalize(phase_vec, dim=-1, eps=1e-6)


model = PhaseTokenTransformer(cfg)
sum(p.numel() for p in model.parameters())

## Loss And Metrics

Training uses `1 - cos(error)` through vector dot product. Reporting uses absolute circular error in degrees, matching the project goal.

In [ ]:
def circular_vector_loss(pred_vec: torch.Tensor, target_vec: torch.Tensor) -> torch.Tensor:
    dot = (pred_vec * target_vec).sum(dim=-1).clamp(-1.0, 1.0)
    return (1.0 - dot).mean()


def phase_from_vec(vec: torch.Tensor) -> torch.Tensor:
    return torch.atan2(vec[:, 1], vec[:, 0])


def circular_abs_error_deg(pred_phase: torch.Tensor, target_phase: torch.Tensor) -> torch.Tensor:
    err = torch.atan2(torch.sin(pred_phase - target_phase), torch.cos(pred_phase - target_phase))
    return torch.rad2deg(torch.abs(err))


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, float]:
    model.eval()
    losses = []
    errors = []
    for batch in loader:
        tokens = batch["tokens"].to(device)
        target_vec = batch["target_vec"].to(device)
        target_phase = batch["target_phase"].to(device)
        pred_vec = model(tokens)
        loss = circular_vector_loss(pred_vec, target_vec)
        pred_phase = phase_from_vec(pred_vec)
        err = circular_abs_error_deg(pred_phase, target_phase)
        losses.append(loss.detach().cpu())
        errors.append(err.detach().cpu())

    all_errors = torch.cat(errors)
    return {
        "loss": float(torch.stack(losses).mean()),
        "mean_abs_error_deg": float(all_errors.mean()),
        "median_abs_error_deg": float(all_errors.median()),
        "p90_abs_error_deg": float(torch.quantile(all_errors, 0.90)),
    }

## Training Loop

Do not run this section while the GPU is occupied. It uses `cuda` when available, which is the AMD Windows PyTorch device path on this machine.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def train_model(cfg: PhaseModelConfig, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=0.01)
    history = []
    step = 0

    while step < cfg.max_steps:
        model.train()
        for batch in train_loader:
            tokens = batch["tokens"].to(device)
            target_vec = batch["target_vec"].to(device)

            pred_vec = model(tokens)
            loss = circular_vector_loss(pred_vec, target_vec)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            step += 1
            if step == 1 or step % cfg.eval_every == 0:
                metrics = evaluate(model, val_loader, device)
                metrics.update({"step": step, "train_loss": float(loss.detach().cpu())})
                history.append(metrics)
                print(metrics)

            if step >= cfg.max_steps:
                break

    return pd.DataFrame(history)


# Run when the GPU is free:
# history = train_model(cfg, model, train_loader, val_loader)

## Prediction Inspection

After training, use this helper to inspect predicted target phase, target nominal phase, and the acausal 180 degree timestamp for validation examples.

In [ ]:
@torch.no_grad()
def collect_predictions(model: nn.Module, loader: DataLoader, device: torch.device, max_batches: int = 4) -> pd.DataFrame:
    model.eval()
    rows = []
    for batch_idx, batch in enumerate(loader):
        if batch_idx >= max_batches:
            break
        tokens = batch["tokens"].to(device)
        pred_vec = model(tokens)
        pred_phase = phase_from_vec(pred_vec).cpu()
        target_phase = batch["target_phase"].cpu()
        error_deg = circular_abs_error_deg(pred_phase, target_phase)
        rows.append(
            pd.DataFrame(
                {
                    "input_time_s": batch["input_time"].cpu().numpy(),
                    "acausal_180_target_time_s": batch["target_time"].cpu().numpy(),
                    "pred_target_nominal_phase_deg": (torch.rad2deg(pred_phase) % 360).numpy(),
                    "true_target_nominal_phase_deg": (torch.rad2deg(target_phase) % 360).numpy(),
                    "abs_error_deg": error_deg.numpy(),
                }
            )
        )
    return pd.concat(rows, ignore_index=True)


# Run after training:
# predictions = collect_predictions(model, val_loader, device)
# predictions.head(20)